# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [3]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ SQL Server

In [4]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Kho= """SELECT Kho_ID, dbo.DecodeUTF8String(Kho) AS Kho, MaxID, Mo FROM Kho """
df_kho = pd.read_sql(query_Kho, conn_libol)

print(df_kho)

   Kho_ID           Kho  MaxID     Mo
0       7            KM      2   True
1       8            NV  71095   True
2       9            GT  61950   True
3       5            KM  15090   True
4       6            KD  10725  False
5      19   Đọc tại chỗ      1  False
6      13  Kho thanh lý      1  False
7      14   Unavailbale      1  False
8      16           CLC    106  False
9      18       Kho Lưu      4  False


C:\Users\admin\AppData\Local\Temp\ipykernel_15208\25397895.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_kho = pd.read_sql(query_Kho, conn_libol)


## Xử lý data

In [5]:
new_row = pd.DataFrame({'Kho_ID': [0], # Tạo hàng dữ liệu giả lập cho kho không xác định
                        'Kho': ['(Không xác định)'],
                        'MaxID': [0], 
                        'Mo': ['False']})
df_kho = pd.concat([df_kho, new_row], ignore_index=True) # Thêm vào dataframe

df_kho = df_kho.sort_values(by="Kho_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_kho)

    Kho_ID               Kho  MaxID     Mo
0        0  (Không xác định)      0  False
1        5                KM  15090   True
2        6                KD  10725  False
3        7                KM      2   True
4        8                NV  71095   True
5        9                GT  61950   True
6       13      Kho thanh lý      1  False
7       14       Unavailbale      1  False
8       16               CLC    106  False
9       18           Kho Lưu      4  False
10      19       Đọc tại chỗ      1  False


## Load data

### [Nếu cần] Clear bảng

In [6]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Kho"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [7]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Kho (ID_kho, Kho, MaxID, Mo) 
                VALUES (?, ?, ?, ?)
               """
for index, row in df_kho.iterrows():
    values = (row['Kho_ID'], 
                row['Kho'],
                row['MaxID'],
                row['Mo'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()